# Module 2: Vector RAG Hallucinates

This notebook compares two agents that answer the same questions from the same hotel data. The vector agent retrieves the three most similar FAQ chunks. The graph agent converts each question into Cypher and queries the full knowledge graph.

The four tests show how those retrieval methods affect the answer.

**Research background:**

- [Internal Representations as Indicators of Hallucinations](https://arxiv.org/pdf/2601.05214)
- [RAG-KG-IL: Multi-Agent Hybrid Framework](https://arxiv.org/pdf/2503.13514)
- [MetaRAG: Metamorphic Testing for Hallucination Detection](https://arxiv.org/pdf/2509.09360)

---

## Compare the Retrieval Methods

| Test | Required operation | Vector agent | Graph agent |
|------|--------------------|--------------|-------------|
| Aggregation | Calculate an average across matching hotels | Reasons from three retrieved chunks | Runs `AVG()` across all matching hotels |
| Counting | Count every matching hotel | Reasons from three retrieved chunks | Runs `COUNT()` across all matching hotels |
| Multiple criteria | Find hotels that satisfy several conditions | Retrieves text by similarity | Traverses relationships and applies each condition |
| No matching data | Detect that the dataset has no answer | Still receives the nearest chunks | Receives an empty query result |

---

## Configure AWS Credentials

This notebook uses Amazon Bedrock as the default model provider for Strands Agents. Configure your AWS credentials before you continue.

To use a different provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).

In [ ]:
import sys, os
# Add notebooks/ root to path so the shared `workshop` package is importable
_notebooks_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _notebooks_root not in sys.path:
    sys.path.insert(0, _notebooks_root)


In [ ]:
import os

# Bedrock needs a region, and botocore reads only AWS_DEFAULT_REGION, never
# AWS_REGION. This sets both from one resolved value so that clients built
# without an explicit region_name land in the workshop's region instead of
# whatever the active AWS profile happens to configure.
from workshop.aws_region import configure_aws_region

configure_aws_region()

# Verify AWS credentials are available
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("\u2705 AWS credentials configured")

# To use OpenAI instead of Bedrock:
#   pip install "strands-agents[openai]"
#   os.environ["OPENAI_API_KEY"] = "your-key-here"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")


## Load the Data and Connect to Services

In [ ]:
import os
os.environ['OTEL_SDK_DISABLED'] = 'true'
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from strands import Agent, tool
from neo4j import GraphDatabase
import faiss
import json
import boto3
import numpy as np

from workshop.aws_region import aws_region
from workshop.bedrock_providers import BEDROCK_CONFIG
from workshop.retrieval_contract import (
    EMBEDDING_DIMENSIONS,
    EMBEDDING_MODEL_ID,
    EMBEDDING_PURPOSE,
)

# Get Neo4j credentials from environment variables
# Workshop Studio: Pre-configured in /etc/environment by CloudFormation
# Self-paced: Set NEO4J_URI and NEO4J_PASSWORD before running
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USER = os.getenv('NEO4J_USERNAME', os.getenv('NEO4J_USER', 'neo4j'))
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

if not NEO4J_URI or not NEO4J_PASSWORD:
    raise ValueError(
        'Neo4j credentials not found.\n'
        'Expected environment variables: NEO4J_URI, NEO4J_PASSWORD\n'
        'Workshop Studio: Check that CloudFormation deployment completed successfully.\n'
        'Self-paced: Set these variables before running the notebook.'
    )

print(f"\u2705 Neo4j URI: {NEO4J_URI}")
print(f"\u2705 Neo4j password: {'*' * len(NEO4J_PASSWORD)} ({len(NEO4J_PASSWORD)} chars)")

# Amazon Bedrock Nova 2 for embeddings (no SentenceTransformer / OpenAI needed)
_bedrock = boto3.client(
    "bedrock-runtime", region_name=aws_region(), config=BEDROCK_CONFIG
)

def _embed(text):
    """Embed text using Amazon Bedrock Nova 2 Multimodal Embeddings."""
    resp = _bedrock.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=json.dumps({
            "taskType": "SINGLE_EMBEDDING",
            "singleEmbeddingParams": {
                "embeddingPurpose": EMBEDDING_PURPOSE,
                "embeddingDimension": EMBEDDING_DIMENSIONS,
                "text": {"truncationMode": "END", "value": text[:8000]},
            },
        }),
        contentType="application/json",
        accept="application/json",
    )
    result = json.loads(resp["body"].read())
    return np.array([result["embeddings"][0]["embedding"]], dtype="float32")

# Build FAISS index if not already built
from pathlib import Path

def build_faiss_if_needed():
    full_index = "faqs_vector.index"
    full_docs = "faqs_docs.json"
    
    if os.path.exists(full_index) and os.path.exists(full_docs):
        return full_index, full_docs
    
    raise FileNotFoundError(
        f"{full_index} and {full_docs} were not found in {os.getcwd()}.\n"
        "Both files (the pre-built FAISS index over the full 300-doc FAQ set) "
        "ship committed with the workshop repository, so a missing file means "
        "your copy of the repository is incomplete.\n"
        "Re-clone the repository or re-download the workshop files, then run "
        "this cell again."
    )

index_file, docs_file = build_faiss_if_needed()

# Load FAISS
index = faiss.read_index(index_file)
with open(docs_file, "r", encoding="utf-8") as f:
    documents = json.load(f)
print(f"\u2705 FAISS: {len(documents)} documents")

# Check Neo4j
from functools import lru_cache

@lru_cache(maxsize=1)
def _get_driver():
    """Create once and reuse the driver across tool calls, mirroring the
    cached-driver pattern in workshop/hybrid_retrieval.py."""
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with _get_driver().session() as session:
    count = session.run('MATCH (h:Hotel) RETURN count(h) as c').single()['c']
    print(f"\u2705 Neo4j: {count} hotels in knowledge graph")

## Create the Tools and Agents

In [ ]:
# Tool observability hook - captures tool inputs and outputs
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import BeforeToolCallEvent, AfterToolCallEvent

class ToolObservabilityHook(HookProvider):
    """Display tool inputs and outputs for debugging and comparison.
    
    Uses Strands hook system to intercept tool execution:
    - BeforeToolCallEvent: captures tool name and input parameters
    - AfterToolCallEvent: captures tool result and status
    
    This approach keeps tools clean (no logging code inside them)
    and provides a reusable observability layer.
    """
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.on_before_tool)
        registry.add_callback(AfterToolCallEvent, self.on_after_tool)
    
    def on_before_tool(self, event: BeforeToolCallEvent) -> None:
        """Called before tool execution - shows input parameters."""
        tool_input = event.tool_use.get("input", {})
        
        print(f"   📥 Tool Input:")
        for key, value in tool_input.items():
            val_str = str(value)
            # Truncate long values (e.g., Cypher queries)
            if len(val_str) > 300:
                val_str = val_str[:300] + "..."
            print(f"      {key}: {val_str}")
    
    def on_after_tool(self, event: AfterToolCallEvent) -> None:
        """Called after tool execution - shows output/result."""
        status = event.result["status"]
        
        # Handle errors
        if status == "error" or event.exception:
            print(f"   ❌ Error: {event.exception}")
            return
        
        # Extract content from result
        if event.result["content"]:
            for content_item in event.result["content"]:
                # Most common: text output
                if "text" in content_item:
                    result_text = content_item["text"]
                    # Show first 800 chars of output
                    if len(result_text) > 800:
                        print(f"   📤 Tool Output (truncated):\n{result_text[:800]}...")
                    else:
                        print(f"   📤 Tool Output:\n{result_text}")
                # Handle JSON output
                elif "json" in content_item:
                    print(f"   📤 Tool Output (JSON):\n{content_item['json']}")

print("✅ Tool observability hook defined")

In [ ]:
@tool
def search_faqs(query: str) -> str:
    """Search hotel FAQs using vector similarity (Traditional RAG)."""
    try:
        query_embedding = _embed(query)
        distances, indices = index.search(query_embedding, 3)
        results = []
        for idx in indices[0]:
            doc = documents[idx]
            results.append(f"[{doc['filename']}]\n{doc['text'][:500]}...")
        return "\n\n".join(results)
    except Exception as e:
        return f"Query error: {str(e)}"

@tool
def query_knowledge_graph(cypher_query: str) -> str:
    """Execute a Cypher query against the hotel knowledge graph.
    
    Node labels: Hotel, Room, Amenity, Policy, Service
    Hotel properties: name, address, guest_rating, total_rooms, email, phone
    Room properties: type, max_occupancy, min_rate, max_rate, bed_configuration
    Amenity properties: name, description
    Policy properties: name, description

    Relationships: (Hotel)-[:HAS_ROOM]->(Room), (Hotel)-[:OFFERS_AMENITY]->(Amenity),
                   (Hotel)-[:HAS_POLICY]->(Policy), (Hotel)-[:PROVIDES_SERVICE]->(Service)

    Location is in Hotel.address property. Use: WHERE h.address CONTAINS 'Cairo'
    IMPORTANT: Property names use snake_case (e.g., guest_rating NOT guestRating, total_rooms NOT totalRooms)
    """
    try:
        with _get_driver().session() as session:
            result = session.run(cypher_query)
            records = list(result)
            if not records:
                return "No results found."
            output = f"Found {len(records)} results:\n"
            for record in records[:15]:
                output += f"  {dict(record.items())}\n"
            return output
    except Exception as e:
        return f"Query error: {str(e)}"

# Create observability hook instance
observability_hook = ToolObservabilityHook()

rag_agent = Agent(
    name="RAG_Agent",
    system_prompt="You are a travel agent. Use vector search to find relevant FAQ information.",
    tools=[search_faqs],
    hooks=[observability_hook],
)

graph_agent = Agent(
    name="GraphRAG_Agent",
    system_prompt="You are a travel agent. Use the knowledge base to answer questions accurately. You can run multiple queries.",
    tools=[query_knowledge_graph],
    hooks=[observability_hook],
)

print("\u2705 Agents ready")

---

## Test 1: Calculate an Average

Vector search returns three similar FAQ chunks. Those chunks do not cover every hotel in Paris, so they cannot support an exact average across the full dataset. The graph agent can ask Neo4j to run `AVG()` across every matching hotel.

**Query:** What is the average guest rating across all hotels in Paris?

### Read Token Usage from Strands

Strands Agents records token usage in `AgentResult.metrics`. Read the accumulated values after each agent call:

```python
result = agent("Your query")
usage = result.metrics.accumulated_usage
print(f"Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")
```

Available metrics:

- `inputTokens`: Tokens sent to the model
- `outputTokens`: Tokens generated by the model  
- `totalTokens`: Total input and output tokens
- `cacheReadInputTokens`: Tokens read from the prompt cache
- `cacheWriteInputTokens`: Tokens written to the prompt cache

The cells below print these values so you can compare token usage between the two agents.

In [ ]:
query = "What is the average guest rating of all hotels in Paris?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
r = rag_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

In [ ]:
query = "What is the average guest rating of all hotels in Paris?"

print("[GRAPH-RAG]")
r = graph_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 Vector agent reasons from 3 documents | Graph agent runs AVG() across all Paris hotels")

---

## Test 2: Count Matching Hotels

The vector agent receives only three FAQ chunks, so those results cannot provide a count across the full dataset. The graph agent can ask Neo4j to run `COUNT()` across every hotel with a swimming pool.

**Query:** How many hotels in the database have a swimming pool?

In [ ]:
query = "How many hotels in the database have a swimming pool?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
r = rag_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

In [ ]:
query = "How many hotels in the database have a swimming pool?"

print("[GRAPH-RAG]")
r = graph_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 Vector agent reasons from 3 documents | Graph agent runs COUNT() across all hotels")

---

## Test 3: Apply Multiple Criteria

Vector search ranks chunks by semantic similarity. It does not enforce that each result matches the city and both required amenities. The graph agent can traverse the hotel and amenity relationships, then apply all three conditions.

**Query:** Which hotels in Cairo have both a spa and a swimming pool, and what are their guest ratings?

In [ ]:
query = "Which hotels in Cairo have both a spa and a swimming pool, and what are their guest ratings?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
r = rag_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

In [ ]:
query = "Which hotels in Cairo have both a spa and a swimming pool, and what are their guest ratings?"

print("[GRAPH-RAG]")
r = graph_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 Vector search ranks individual chunks | Graph query applies the city, spa, and pool filters")

---

## Test 4: Handle a Question with No Matching Data

The dataset contains no hotels in Antarctica. Vector search still returns the three nearest chunks, even when they are irrelevant to the question. The model may use those chunks to produce an unsupported answer. A Neo4j query returns no matching records, which gives the graph agent a clear signal that the data has no answer.

**Query:** Tell me about hotels in Antarctica

In [ ]:
query = "Tell me about hotels in Antarctica"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
r_rag = rag_agent(query)

if r_rag.metrics:
    usage = r_rag.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

In [ ]:
query = "Tell me about hotels in Antarctica"

print("\n[GRAPH-RAG]")
r_graph = graph_agent(query)

if r_graph.metrics:
    usage = r_graph.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 Vector search returns the nearest documents | Graph query returns no matching hotels")